# Wine Quality Analysis & Prediction

**Name:** Heramb Manglani

## Objective
The aim of this project is to explore the Wine Quality dataset, understand which chemical properties are related to wine quality, and build a machine-learning model to predict the quality score.

## 1. Getting the Dataset Ready

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Read the dataset
file_name = "Wine Quality Dataset.csv"
wine = pd.read_csv(file_name)

print("Dataset imported successfully.")
print("Rows:", wine.shape[0])
print("Columns:", wine.shape[1])

wine.head()

## 2. Understanding the Data

In [ ]:
wine.info()

print("\nColumn names:")
for col in wine.columns:
    print("-", col)

print("\nDescriptive statistics:")
wine.describe().T

## 3. Data Quality Check

In [ ]:
missing = wine.isna().sum()

print("Missing values by column:")
print(missing)

print("\nDuplicate rows:", wine.duplicated().sum())

There are no missing values in the supplied dataset, so there is no need for imputation. I also checked for duplicate records before moving to the modelling stage.

## 4. Exploring Wine Quality

In [ ]:
quality_counts = wine["quality"].value_counts().sort_index()

print("Number of wines at each quality level:")
print(quality_counts)

plt.figure(figsize=(7, 4))
sns.countplot(data=wine, x="quality", hue="quality", legend=False)
plt.title("Wine Quality Distribution")
plt.xlabel("Quality score")
plt.ylabel("Number of samples")
plt.tight_layout()
plt.show()

The quality scores are concentrated around the middle values. Scores 5 and 6 make up a large part of the dataset, while the extreme scores occur much less often. This imbalance is important when interpreting classification results.

## 5. Looking at Relationships Between Variables

In [ ]:
correlations = wine.corr(numeric_only=True)

plt.figure(figsize=(11, 8))
sns.heatmap(correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

print("Correlation with quality:")
print(correlations["quality"].sort_values(ascending=False))

From the correlation values, alcohol has the strongest positive relationship with the quality score, while density has a comparatively strong negative relationship. Correlation alone does not prove that one variable causes quality to change, so the variables are considered together when building the model.

## 6. Preparing Data for Machine Learning

In [ ]:
# Separate predictors from the target
X = wine.drop(columns="quality")
y = wine["quality"]

# Keep the class proportions similar in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=7,
    stratify=y
)

# Scale the numerical features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

I used an 80:20 train-test split. Stratification was included because the quality classes are not evenly distributed. Standardisation was then applied to the numerical predictors using statistics learned only from the training data.

## 7. Building the Prediction Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=250,
    max_depth=None,
    min_samples_leaf=2,
    random_state=7,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)

print("Random Forest model trained successfully.")

## 8. Evaluating the Model

In [ ]:
accuracy = accuracy_score(y_test, predictions)

print(f"Test accuracy: {accuracy:.3f}")
print("\nDetailed classification report:")
print(classification_report(y_test, predictions, zero_division=0))

In [ ]:
labels = sorted(y.unique())
matrix = confusion_matrix(y_test, predictions, labels=labels)

plt.figure(figsize=(7, 5))
sns.heatmap(
    matrix,
    annot=True,
    fmt="d",
    xticklabels=labels,
    yticklabels=labels,
    cmap="Blues"
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted quality")
plt.ylabel("Actual quality")
plt.tight_layout()
plt.show()

The accuracy gives an overall view of the predictions, while the classification report shows how the model behaves for each quality level. The confusion matrix makes it easier to see which quality scores are being confused with one another. Rare classes should be interpreted carefully because there are relatively few examples available for training.

## 9. Which Features Matter Most?

In [ ]:
importance = (
    pd.Series(model.feature_importances_, index=X.columns)
      .sort_values(ascending=True)
)

plt.figure(figsize=(8, 6))
importance.plot(kind="barh")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

print("Features ranked by importance:")
print(importance.sort_values(ascending=False))

Feature importance shows how useful each predictor was to the Random Forest's decision process. This is a model-specific measure, so it should not be interpreted as a direct causal effect on wine quality.

## 10. Final Findings

In [ ]:
top_feature = corr.drop("quality").idxmax()
lowest_feature = corr.drop("quality").idxmin()

print("Dataset size:", wine.shape)
print("Missing values:", missing_total)
print("Most positively correlated feature:", top_feature)
print("Most negatively correlated feature:", lowest_feature)
print(f"Model accuracy on the test set: {accuracy:.3f}")

### Conclusion

The dataset contains 4,898 wine observations and 11 input variables used to describe the wines. The target variable, `quality`, is not evenly distributed, with most observations concentrated around the middle quality scores.

The exploratory analysis shows that alcohol has the clearest positive correlation with quality, while density shows a notable negative correlation. However, the prediction task depends on several variables together rather than one measurement alone.

A Random Forest classifier was trained after splitting and standardising the data. The final accuracy and class-level metrics above show how well the model performs on unseen observations. The confusion matrix also highlights the effect of the less-represented quality classes.

### Possible next steps

- Compare Random Forest with other classification algorithms.
- Experiment with class-weighting or resampling for the less common quality scores.
- Try regression because wine quality is an ordered numerical score.
- Tune the model's hyperparameters and compare the results.